In [0]:
CREATE OR REPLACE TEMP VIEW sample_employee_logs AS
SELECT * FROM VALUES
  (101, 'Karyawan mengakses repository kode rahasia di luar jam kerja dari IP address tidak dikenal sebanyak 15 kali berturut-turut.'),
  (102, 'Lead Data Engineer mengunduh skema kompensasi dan gaji seluruh eksekutif setelah pengunduran dirinya ditolak.'),
  (103, 'Senior developer menyelesaikan pelatihan tahunan data security tepat waktu dengan skor sempurna.')
AS t(employee_id, incident_text);

Step 1: Menguji ai_classify()
Fungsi ini digunakan untuk mengelompokkan teks tidak terstruktur ke dalam daftar label tertentu secara otomatis tanpa perlu menulis prompt manual.

In [0]:
SELECT 
  employee_id,
  incident_text,
  ai_classify(incident_text, array('Low Risk', 'Medium Risk', 'Critical Risk')) AS evaluated_risk_level
FROM sample_employee_logs;

employee_id,incident_text,evaluated_risk_level
101,Karyawan mengakses repository kode rahasia di luar jam kerja dari IP address tidak dikenal sebanyak 15 kali berturut-turut.,Critical Risk
102,Lead Data Engineer mengunduh skema kompensasi dan gaji seluruh eksekutif setelah pengunduran dirinya ditolak.,Critical Risk
103,Senior developer menyelesaikan pelatihan tahunan data security tepat waktu dengan skor sempurna.,Low Risk


Step 2: Menguji ai_extract()
Fungsi ini digunakan untuk menarik entitas, kata kunci, atau variabel spesifik dari paragraf teks ke dalam format data terstruktur.

In [0]:
SELECT 
  employee_id,
  ai_extract(
    incident_text, 
    array('job_role', 'unauthorized_action', 'frequency')
  ) AS extracted_entities
FROM sample_employee_logs
WHERE employee_id = 101;

employee_id,extracted_entities
101,"List(Karyawan, mengakses repository kode rahasia di luar jam kerja dari IP address tidak dikenal, 15 kali berturut-turut)"


Step 3: Menguji ai_summarize()
Fungsi ini berguna untuk memadatkan teks panjang, log audit, atau laporan evaluasi performa talenta menjadi ringkasan singkat.

In [0]:
SELECT 
  employee_id,
  ai_summarize(incident_text, 10) AS short_executive_summary
FROM sample_employee_logs;
-- WHERE employee_id = 102;

employee_id,short_executive_summary
101,Akses kode rahasia tidak sah
102,Eksekutif unduh skema gaji
103,Senior developer lulus pelatihan keamanan data


Step 4: Menguji ai_parse_document()
Fungsi ini dirancang khusus untuk membaca dokumen tidak terstruktur seperti berkas PDF atau gambar (misal: formulir audit bertanda tangan, surat kontrak kerja, atau CV talenta) yang tersimpan di Unity Catalog Volumes.

In [0]:
SELECT 
  path,
  ai_parse_document(content) AS parsed_data
FROM read_files(
  '/Volumes/training_medallion/00_landing/employee_documents/Incident Report 101 - Security & Compliance Audit.pdf',
  format => 'binaryFile'
);

path parsed_data dbfs:/Volumes/training_medallion/00_landing/employee_documents/Incident Report 101 - Security & Compliance Audit.pdf {"document":{"elements":[{"bbox":[{"coord":[137,140,943,169],"page_id":0}],"confidence":1,"content":"ENTERPRISE CYBERSECURITY & COMPLIANCE DIRECTORATE","description":null,"id":0,"type":"page_header"},{"bbox":[{"coord":[1173,150,1387,211],"page_id":0}],"confidence":1,"content":"CLASSIFICATION:\nCRITICAL","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[139,183,1028,304],"page_id":0}],"confidence":1,"content":"Security Incident & Compliance Audit\nReport","description":null,"id":2,"type":"title"},{"bbox":[{"coord":[139,351,1516,492],"page_id":0}],"confidence":0.9896,"content":" INCIDENT REFERENCE SUBJECT ID / ROLE AUDIT TIMESTAMP DETECTION SOURCE INC-2026-101 EMP-101 (Senior Data Engineer) 2026-10-25 03:42:18 UTC Automated DLP & SIEM Sensor ","description":null,"id":3,"type":"table"},{"bbox":[{"coord":[162,520,654,552],"page_id":0}],"confidence":1,"content":"EXECUTIVE INCIDENT SYNOPSIS","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[175,595,1445,775],"page_id":0}],"confidence":1,"content":"An employee identified under ID EMP-101 initiated multiple unauthorized access requests to the confidential\ncorporate source code and schema repository outside normal business hours. Telemetry logs captured 15\nconsecutive repository clone and branch extraction attempts originating from an unapproved external IP\naddress. The telemetry triggers were flagged as a potential credential compromise or deliberate exfiltration\npattern.","description":null,"id":5,"type":"text"},{"bbox":[{"coord":[162,826,965,861],"page_id":0}],"confidence":1,"content":"FORENSIC EVENT LOG SUMMARY (SAMPLE WINDOW)","description":null,"id":6,"type":"section_header"},{"bbox":[{"coord":[139,878,1516,1320],"page_id":0}],"confidence":0.9923,"content":" Timestamp (UTC) Target Asset Origin IP Address Action Type Status 03:21:04 repo-prod-core-engine.git 198.51.100.44 (External) SSH Key Handshake & Auth Flagged 03:22:15 repo-prod-core-engine.git 198.51.100.44 (External) Git Clone (Archive Payload) Flagged 03:24:50 sec-credentials-vault-ro 198.51.100.44 (External) Bulk Read Secret Tokens Blocked 03:31:12 talent-compensation-db.dw 198.51.100.44 (External) Schema Export Query Denied ","description":null,"id":7,"type":"table"},{"bbox":[{"coord":[162,1346,899,1379],"page_id":0}],"confidence":1,"content":"POLICY VIOLATIONS & REMEDIATION PROTOCOL","description":null,"id":8,"type":"section_header"},{"bbox":[{"coord":[167,1421,498,1454],"page_id":0}],"confidence":1,"content":"Compliance Policy Violations","description":null,"id":9,"type":"section_header"},{"bbox":[{"coord":[167,1463,745,1667],"page_id":0}],"confidence":1,"content":"SEC-POL-04: Unauthorized off-hours access from\nuntrusted external network networks.\nDLP-POL-12: Direct cloning of tier-1 intellectual\nproperty to unmanaged hardware.\nAUTH-CTR-09: Anomalous credential re-use across\ndistinct administrative zones.","description":null,"id":10,"type":"text"},{"bbox":[{"coord":[864,1421,1231,1454],"page_id":0}],"confidence":1,"content":"Mandatory Corrective Directives","description":null,"id":11,"type":"section_header"},{"bbox":[{"coord":[867,1463,1480,1667],"page_id":0}],"confidence":1,"content":"Immediate revocation of IAM session keys and\nassociated SSH credentials.\nDevice isolation protocol enacted on target workstation\nterminal.\nEscalation to HR Talent Ethics Board for formal\nemployee inquiry within 24h.","description":null,"id":12,"type":"text"},{"bbox":[{"coord":[162,1721,566,1753],"page_id":0}],"confidence":1,"content":"VERIFICATION & SIGN-OFF","description":null,"id":13,"type":"section_header"},{"bbox":[{"coord":[169,1796,420,1821],"page_id":0}],"confidence":1,"content":"INVESTIGATING AUDITOR","description":null,"id":14,"type":"section_header"},{"bbox":[{"coord":[169,1826,369,1892],"page_id":0}],"confidence":1,"content":"Sarah Chen, CISSP\nSigned 

End-to-End Pipeline Testing (Function Chaining)

In [0]:
SELECT 
  employee_id,
  ai_classify(incident_text, array('Low Risk', 'Medium Risk', 'Critical Risk')) AS risk_tier,
  ai_summarize(incident_text, 10) AS brief_summary,
  ai_extract(incident_text, array('job_role', 'frequency')) AS extracted_metadata
FROM sample_employee_logs;

employee_id,risk_tier,brief_summary,extracted_metadata
101,Critical Risk,Akses kode rahasia tidak sah,"List(Karyawan, 15 kali berturut-turut)"
102,Critical Risk,Eksekutif unduh skema gaji,"List(Lead Data Engineer, null)"
103,Low Risk,Senior developer lulus pelatihan keamanan data,"List(Senior developer, annual)"
